<h1>Chapter 6 - Planning & Reflection</h1>
<i>Autonomy for your `TinyAgent` through Native Tool Calling and Reasoning</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 6 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [16]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [17]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM - `Gemma 3`

At the beginning of every chapter, we start by choosing the LLM that we want to use. In this notebook, we will explore how to enable autonomous behavior for LLMs that do not have native tool calling and reasoning behavior. As such, the model that we will be using throughout this chapter is Gemma 3, a model that does not have native tool calling capabilities.

In [18]:
import os
from illustrated_agents.chapters.ch5_native import LLM  # We use our most recent version of `LLM`

# Ollama through OpenAI API
llm = LLM(model="gemma3:12b", backend="openai", api_base="http://localhost:11434/v1/")

# Llama.cpp server
# llm = LLM(model="openai/gemma-3-12B-it-Q4_K_M", backend="litellm", api_base="http://localhost:8080")

# LM Studio
# llm = LLM(model="lm_studio/gemma-4-12B-it", backend="litellm", api_base="http://localhost:1234/v1")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash", backend="litellm", api_key=None)

## 2 - Adding **`Autonomy`**

In the previous chapter, we added the `Tools` module to your `TinyAgent`. In this chapter, we will cover how to give it more autonomy:

![../images/ch6.png](../images/ch6.png)

Throughout this notebook we will implement a module that allows for action sequencing, namely Reason and Act (ReAct). Then, we add a reflection module that will check whether the `TinyAgent` is doing the right thing and adjust accordingly.

## 3 - Reason and Act (ReAct)

Autonomy, as covered in the book, is typically achieved through some method of task decomposition or planning. We are going to use the ReAct framework to give this autonomy to your `TinyAgent`. To do so, we are going to implement loops of:

* `THOUGHT` - A reasoning step about the current situation
* `ACTION` - An action to execute (e.g., a tool)
* `OBSERVATION` - A generated observation (typically the output of a tool)

Note that in this example, the `OBSERVATION` will be provided by us and not the LLM. This allows for a more simplified example as it would otherwise need quite a bit of code to add this observation. 

ReAct is mostly about prompting the LLM, so that's what we are going to do! The `ReAct` class consists of three functions:

* `system_prompt` - The ReAct prompt
* `parse_react` - Parse the LLM generated `THOUGHT` and `ACTION` so that they are separated for tool usage
* `format_observation` - Create the `OBSERVATION` based on the output of the tool

We instruct the `TinyAgent` to always use an `ACTION` and that if it does not need a tool, it can simply return it as:

* `"final_answer"` - The final answer as text that **stops** the `TinyAgent` from running


In [2]:
import re

class ReAct:
    """ReAct module."""

    def __init__(self, max_steps: int = 10):
        """Initialize ReAct module.

        Arguments:
            max_steps: Maximum number of ReAct steps to perform.
        """
        self.max_steps = max_steps

    @property
    def prompt(self) -> str:
        return """
# ReAct (Reason and Act)

You are a ReAct agent that performs exactly ONE step per turn.

## ReAct Format

You use the following format for each step:

THOUGHT: [Your reasoning about what to do next]
ACTION:
{
    "tool": "a_tool_name",
    "kwargs": {"param": "value"},
}

An observation will be provided after each action. You do not generate the observation yourself.

## ReAct Completion

To provide the final answer to the task, use an action blob with "tool": "final_answer" tool. 
It is the only way to complete the task, else you will be stuck on a loop. 
So your final output should look like this:

ACTION:
{
    "tool": "final_answer",
    "kwargs": "insert your final answer here"
}

Use the `final_answer` tool when you are completely done with all subtasks and have the final answer ready.
You can also use `final_answer` to directly reply to a user's question without using any other tools.
"""

    def parse(self, response) -> str:
        """Parse a ReAct formatted response into THOUGHT and ACTION."""
        text = response.content

        # The patterns for each section
        patterns = {
            "THOUGHT": r"THOUGHT:\s*(.+?)(?=ACTION:|OBSERVATION:|$)",
            "ACTION": r"ACTION:\s*(.+?)(?=THOUGHT:|OBSERVATION:|$)",
        }

        # Extract each section using regex
        result = {}
        for key, pattern in patterns.items():
            match = re.search(pattern, text, re.DOTALL)
            result[key] = match.group(1).strip() if match else ""

        # Update Response and extract only the action
        response.content = result["ACTION"]
        response.reasoning = result["THOUGHT"]
        return response

There is a lot happening there, so let's go through each function step-by-step starting with the prompt. Since ReAct is mostly a prompting technique, the prompt in itself is the most important step. The prompt, which we crafted through careful trial and error, has several components that describe how your `TinyAgent` should behave. Below, we have annotated this function for you so it is clear what the highlights are. Note that we might update prompt slight, but the general idea should remain the same:


In [3]:
from illustrated_agents.chapters.ch6 import react_prompt_annotated; react_prompt_annotated

The `parse_react` function is needed to parse the strings that contain THOUGHT and ACTION into separate entities:

In [4]:
from illustrated_agents.chapters.ch6 import react_parse_annotated; react_parse_annotated

---

💡 **NOTE 1**: For execution, we only return the `ACTION` section since that's what we need to execute tools and provide answers.   
The THOUGHT section is useful for interpretability but not needed for execution. 

💡 **NOTE 2**: Use regular expressions (`re`) is definitely not the most stable way of doing this! What if the LLM makes a small mistake and uses "*THOUGH:*" instead of "*THOUGHT:*"? This will not capture and that is on purpose. We want to showcase the most minimal way of approaching this and the downsides of doing this. Likewise, we could construct a 100+ line function/class that is more robust but defeats the purpose of an educational example. That said, this downside also nicely demonstrates why native tool calling is more robust since the LLM has seen very specific tokens it can use to perform tool calling. This drastically reduces the error rate but makes an Agent a bit more of a blackbox. Don't worry, we're definitely also showing you how to do this in `chapter06_native_react.ipynb`.

---

## 4 - Updating `agent.py`

There are several changes needed to `TinyAgent` that requires changes throughout the class. In particular:

* `__init__` -- We need to add the prompt of the `ReAct` to the system prompt
* `run` -- Your `TinyAgent` now runs for a number of steps, defined by `ReAct`
* `_step` -- Now performs a single step of the `ReAct` loop (**THOUGHT**/**ACTION**/**OBSERVATION**)
* `_execute_action` -- A function to parse the tool call from the **ACTION** step.

In [5]:
from illustrated_agents.chapters.ch2 import Response
from illustrated_agents.chapters.ch5_native import Tools, LLM, Memory


class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory, tools: Tools, planner: ReAct):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = planner
        self.reflector = None  # Chapter 6: Add Reflection
        self.skills = None  # Chapter 6: Add Skills

        # Build system prompt with all components
        system_prompt = "You are a helpful AI agent.\n\n"
        system_prompt += self.planner.prompt + "\n\n"
        system_prompt += self.tools.prompt
        self.memory.add("user", system_prompt)

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task)

        # `Autonomy` loop
        for step in range(self.planner.max_steps):
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

    def _step(self) -> str | None:
        """Perform a single step."""
        # Generate response and add to memory
        response = self.llm.generate(self.memory.get_messages(), tools=self.tools.schemas)
        self.memory.add("assistant", response.content, tool_call=response.tool_call)

        # Parse planner's response to extract action if needed
        response = self.planner.parse(response)

        # Tool parsing and execution
        if self.tools.has_tool_call(response):
            return self._execute_action(response)

        return None

    def _execute_action(self, response: Response) -> str | None:
        """Execute a tool action."""
        tool_call = self.tools.parse_tool_call(response)

        # Final answer ends the loop
        if tool_call["tool"] == "final_answer":
            return tool_call.get("kwargs", "")

        # Execute tool and extract the observation
        observation = self.tools.run_tool(tool_call)
        obs_prompt = f"OBSERVATION: {response.content} -> {observation}"

        # Native tool calling should get the role `tool`
        if self.tools.schemas:
            self.memory.add("tool", str(observation))
        else:
            self.memory.add("user", f"OBSERVATION: {observation}")

        return None

Although we can show the diff like we did before, this time there are so many changes! Instead, let's go through each function and describe how they are used, starting with the `__init__`:

In [6]:
from illustrated_agents.chapters.ch6 import tinyagent_init_annotated; tinyagent_init_annotated

Next, let's explore the `run` function where two interesting actions take place:

In [7]:
from illustrated_agents.chapters.ch6 import tinyagent_run_annotated; tinyagent_run_annotated

Next, the `_step` is where much of the processing happens and now shows an interesting structure:

In [8]:
from illustrated_agents.chapters.ch6 import tinyagent_step_annotated; tinyagent_step_annotated

Finally, the `_execute_action` processes the **ACTION** step:

In [9]:
from illustrated_agents.chapters.ch6 import tinyagent_action_annotated; tinyagent_action_annotated

We can also show it all the changes at once. Compared to previous chapters, the difference is a bit bigger as a result of the `for` loop:

In [10]:
from illustrated_agents.chapters.ch6 import tinyagents_diff; tinyagents_diff

Now that we went through all steps, let's start creating your ReAct-based `TinyAgent` and explore whether it can actually do things autonomously!

In [11]:
# Define tools
def calculator(a: str, b: str) -> float:
    return float(a) + float(b)

def get_weather(location: str) -> str:
    return f"Weather in {location}: Sunny, 72°F"

# Register tools
tools = Tools()
tools.add_tool("calculator", calculator, "Adds two numbers: calculator(a: str, b: str)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(location: str)")

# Memory
memory = Memory()

# ReAct
react = ReAct(max_steps=10)

# Create agent
agent = TinyAgent(llm=llm, tools=tools, memory=memory, planner=react)

In [ ]:
# Multi-step task with reasoning
agent.run("""
I'm planning a trip! Help me with these tasks:
1. What's the weather in New York City?
2. What's the weather in Los Angeles?
3. I saved $150.50 and my friend is giving me $75.25. How much do I have for the trip?

Based on the weather, which city would you recommend I visit?
""")

Amazing! If you used **Gemma 3 12B** then it should have run correctly. It performed all tasks in sequence and gave us back the final reply. Let's explore how many steps it took:

In [14]:
from rich import print as pprint

# Extract metadata
messages = agent.memory.get_messages()
nr_api_calls = sum(1 for msg in messages if "ACTION:" in msg["content"] and '"tool":' in msg["content"] and '"final_answer"' not in msg["content"])
nr_assistant_msgs = sum(1 for msg in messages if msg["role"] == "assistant")

# Show output and metadata
pprint(f"It took {nr_api_calls} Tool calls and {nr_assistant_msgs} assistant messages!\n\n===== Messages: =====")
pprint(messages)

It took 3 Tool calls and 4 assistant messages!

===== Messages: =====

[
    {
        'role': 'user',
        'content': 'You are a helpful AI agent.\n\n\n# ReAct (Reason and Act)\n\nYou are a ReAct agent that 
performs exactly ONE step per turn.\n\n## ReAct Format\n\nYou use the following format for each step:\n\nTHOUGHT: 
[Your reasoning about what to do next]\nACTION:\n{\n    "tool": "a_tool_name",\n    "kwargs": {"param": 
"value"},\n}\n\nAn observation will be provided after each action. You do not generate the observation 
yourself.\n\n## ReAct Completion\n\nTo provide the final answer to the task, use an action blob with "tool": 
"final_answer" tool. \nIt is the only way to complete the task, else you will be stuck on a loop. \nSo your final 
output should look like this:\n\nACTION:\n{\n    "tool": "final_answer",\n    "kwargs": "insert your final answer 
here"\n}\n\nUse the `final_answer` tool when you are completely done with all subtasks and have the final answer 
ready.\nYou can also use `final_answer` to directly reply to a user\'s question without using any other 
tools.\n\n\n\n# Tools\n\nIf needed, you can only use the following tools to assist you in completing 
tasks:\n\n`calculator`: Adds two numbers: calculator(a: str, b: str)\n`get_weather`: Gets weather: 
get_weather(location: str)\n\nTo use a tool, respond with JSON: {"tool": "name", "kwargs": {"param": "value"}}\n'
    },
    {
        'role': 'user',
        'content': "\nI'm planning a trip! Help me with these tasks:\n1. What's the weather in New York City?\n2. 
What's the weather in Los Angeles?\n3. I saved $150.50 and my friend is giving me $75.25. How much do I have for 
the trip?\n\nBased on the weather, which city would you recommend I visit?\n"
    },
    {
        'role': 'assistant',
        'content': 'THOUGHT: I need to first find out the weather in New York City.\nACTION:\n{\n    "tool": 
"get_weather",\n    "kwargs": {"location": "New York City"}\n}\n'
    },
    {'role': 'user', 'content': 'OBSERVATION: Weather in New York City: Sunny, 72°F'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: Now I need to find out the weather in Los Angeles.\nACTION:\n{\n    "tool": 
"get_weather",\n    "kwargs": {"location": "Los Angeles"}\n}\n'
    },
    {'role': 'user', 'content': 'OBSERVATION: Weather in Los Angeles: Sunny, 72°F'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: Next, I need to calculate the total amount of money I have saved.\nACTION:\n{\n    
"tool": "calculator",\n    "kwargs": {"a": "150.50", "b": "75.25"}\n}\n'
    },
    {'role': 'user', 'content': 'OBSERVATION: 225.75'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: The weather in both cities is the same, sunny and 72°F. I have $225.75 for the trip. 
Since the weather is the same, the choice is arbitrary. I\'ll pick New York City.\nACTION:\n{\n    "tool": 
"final_answer",\n    "kwargs": "New York City"\n}'
    }
]

In our example, 3 tools were called, namely the `get_weather` tool twice and the `calculator` once. Both without any problems. Then, your `TinyAgent` created a final message to summarize the results.

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered what makes your `TinyAgent` autonomous... a for-loop! This for-loop, driven by the Reason and Act (ReAct) framework is quite capable. As we covered in the book, ReAct is not something we always will do explicitly because newer models tend to be trained on that already. Going through the steps of it helps you understand what the LLM is doing under the hood that makes this all possible. 

In [15]:
from illustrated_agents.chapters.ch6 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py    ← Updated (Autonomy with a for-loop ;) ...)                                                     │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py ← New (Added the ReAct (Reason and Act) framework)                                              │
│ ├── toolbox.py                                                                                                  │
│ └── tools.py                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

...